# 03. PySATL mpest Initializers API

## Section 1. Orientation

The estimators of `pysatl_mpest` (`EM`, `ECM`, `ECME`) improve a mixture step by step, but they need a mixture to start
from. A bad start costs iterations at best and ends in a poor local optimum at worst. The initializers API answers the
question that comes before fitting: given a sample and the families of the components, where should the estimator
start?

## What in this notebook

By the end of this notebook you should be able to answer four practical questions.

1. How to get a starting mixture from a sample and a list of families, and how to make it reproducible.
2. What the parts of the cluster initializer do — partitioning, estimation of a component, matching — and when to
   replace one of them.
3. Which failures of an initializer are a bad draw that another run may fix, and which are errors of the input.
4. How to repeat an initializer and keep the best run, and how to compare runs by the result of an estimator.

## Interface overview

Everything starts from `Initializer`: `initialize(X, families, random_state)` returns a `MixtureModel` with one
component per family, in the order of `families`. The library has three initializers.

- `ClusterInitializer` splits the sample into clusters (a `Partitioner`), estimates every family on every cluster
  (a `ComponentEstimator`) and matches the clusters with the families (a `Matcher`).
- `MultiStartInitializer` runs another initializer several times with independent generators and keeps the start with
  the best log-likelihood.
- `RefiningInitializer` takes the start of another initializer and refines it with a mixture estimator, so that a
  multistart can compare runs by the result of the estimator.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
from pysatl_core.families import ParametricFamilyRegister
from pysatl_core.types import FamilyName

from pysatl_mpest.core import MixtureModel
from pysatl_mpest.estimators import EM
from pysatl_mpest.estimators.iterative import StepBreakpointer
from pysatl_mpest.optimizers import ScipyNelderMead

normal = ParametricFamilyRegister.get(FamilyName.NORMAL)
expon = ParametricFamilyRegister.get(FamilyName.EXPONENTIAL)
uniform = ParametricFamilyRegister.get(FamilyName.CONTINUOUS_UNIFORM)


def show(mixture: MixtureModel) -> None:
    for weight, component in zip(mixture.weights, mixture):
        parameters = {name: round(value, 3) for name, value in component.parameters.items()}
        print(f"weight={weight:.3f}  {component.family_name} {parameters}")


def em(steps: int) -> EM:
    return EM(breakpointers=[StepBreakpointer(steps)], pruners=[], optimizer=ScipyNelderMead())


def plot_mixtures(ax, X, mixtures, bins=60):
    grid = np.linspace(X.min(), X.max(), 1000)
    ax.hist(X, bins=bins, density=True, color="lightgray")
    for label, mixture in mixtures.items():
        ax.plot(grid, np.exp(mixture.lpdf(grid)), label=label)
    ax.legend()

from sklearn.cluster import AgglomerativeClustering, KMeans

from pysatl_mpest.estimators import ECM
from pysatl_mpest.estimators.iterative import PriorThresholdPruner
from pysatl_mpest.initializers import (
    ClusterInitializer,
    ComponentEstimationError,
    HungarianMatcher,
    IdentityMatcher,
    InitializationError,
    Initializer,
    KMeansPartitioner,
    MLEComponentEstimator,
    MultiStartInitializer,
    Partitioner,
    QuantilePartitioner,
    RandomCentersPartitioner,
    RefiningInitializer,
    SklearnPartitioner,
)

## Section 2. The running example

One sample stays with us through the notebook: two normal modes and an exponential one. The exponential component
starts at zero and overlaps the right normal mode, so the families matter: a mixture of three normals would describe
this sample worse.

In [ ]:
truth = MixtureModel(
    [normal(mu=-8, sigma=1), normal(mu=-3, sigma=0.7), expon(lambda_=0.5)], weights=[2 / 7, 2 / 7, 3 / 7]
)
X = truth.generate(size=700, random_state=42)
families = [normal, normal, expon]

fig, ax = plt.subplots(figsize=(8, 3.5))
plot_mixtures(ax, X, {"true mixture": truth})
ax.set_title("The running example")
plt.show()

**Summary of Section 2:** the sample `X` has 700 points from a known mixture; the families are
`[normal, normal, expon]`. Every initializer below gets these two arguments.

## Section 3. The Initializer interface

### 3.1 initialize()

`initialize(X, families, random_state=None)` is the whole public interface. The result has one component per family, in
the order of `families`, and weights that sum to one. Here is the cluster initializer with k-means.

In [ ]:
initializer = ClusterInitializer(KMeansPartitioner())
start = initializer.initialize(X, families, random_state=42)
show(start)
print("log-likelihood of the start:", round(start.loglikelihood(X), 2))
print("log-likelihood of the truth:", round(truth.loglikelihood(X), 2))

### 3.2 The order of the components

Component `j` always comes from `families[j]`: settings of the estimators that refer to a component by its number
(`parameter_groups` of `ECM`, strategies of `ECME`) rely on it. Identical families are ordered by the mean of their
clusters, so with `[normal, normal, normal]` component 0 is the leftmost mode in every run.

In [ ]:
reordered = initializer.initialize(X, [expon, normal, normal], random_state=42)
show(reordered)

assert [component.family_name for component in reordered] == ["Exponential", "Normal", "Normal"]
assert reordered[1].parameters["mu"] < reordered[2].parameters["mu"]

### 3.3 Randomness

Randomness is an argument of the call, not a field of the initializer, as in `MixtureModel.generate`:

- an integer seed gives the same mixture however many calls came before;
- a `numpy.random.Generator` is used in place and advances, so two calls with one generator differ;
- `None` takes fresh entropy.

Random centers show the difference best: they pick the centers of the clusters at random.

In [ ]:
random_initializer = ClusterInitializer(RandomCentersPartitioner())
same = [random_initializer.initialize(X, families, random_state=7) for _ in range(2)]
print("integer seed, two calls equal:", same[0] == same[1])

generator = np.random.default_rng(7)
first = random_initializer.initialize(X, families, random_state=generator)
second = random_initializer.initialize(X, families, random_state=generator)
print("one generator, two calls equal:", first == second)

assert same[0] == same[1] and first != second

**Summary of Section 3:** an initializer is a function of the sample, the families and the random state. The order of
the components follows `families`, identical families go from left to right, and an integer seed makes the result
reproducible.

## Section 4. ClusterInitializer

`ClusterInitializer(partitioner, *, component_estimator=None, matcher=None, min_cluster_size=None)` builds a start in
three steps: split the sample into `k = len(families)` clusters, estimate every family on every cluster, and choose
which cluster goes to which family. Each step is a replaceable role.

### 4.1 Partitioners

A `Partitioner` returns a hard partition: a matrix `H` of shape `(n, k)` with exactly one 1 in a row. The library has
four of them.

- `KMeansPartitioner()` — k-means of `scipy`; random initial centers.
- `QuantilePartitioner()` — `k` consecutive segments of the sorted sample with equal numbers of points; deterministic.
- `RandomCentersPartitioner()` — `k` distinct points of the sample as centers, every point to the nearest one; a cheap
  source of different starts for a multistart.
- `SklearnPartitioner(clusterer)` — any clusterer of scikit-learn with `n_clusters`, for example agglomerative or
  spectral clustering.

In [ ]:
partitioners = {
    "k-means": KMeansPartitioner(),
    "quantile": QuantilePartitioner(),
    "random centers": RandomCentersPartitioner(),
    "sklearn KMeans": SklearnPartitioner(KMeans(n_init=1)),
    "agglomerative": SklearnPartitioner(AgglomerativeClustering()),
}
for name, partitioner in partitioners.items():
    H = partitioner.partition(X, 3, np.random.default_rng(42))
    print(f"{name:15} H.shape={H.shape}  cluster sizes={np.count_nonzero(H, axis=0)}")

### 4.2 Estimating a component on a cluster

A `ComponentEstimator` estimates the free parameters of one family on one cluster. The default
`MLEComponentEstimator` uses closed-form maximum likelihood: the mean and the standard deviation for `Normal`, the
inverse mean for `Exponential`, the minimum and the maximum for `ContinuousUniform`. `supports(family)` says whether a
family can be estimated; the initializer checks every family before partitioning.

In [ ]:
estimator = MLEComponentEstimator()
H = KMeansPartitioner().partition(X, 3, np.random.default_rng(42))
cluster = H[:, 0]
print("supports normal:", estimator.supports(normal), " supports normal(meanVar):",
      estimator.supports(normal.view(parametrization_name="meanVar")))
print("normal on cluster 0:", estimator.estimate(X, cluster, normal))
print("exponential on cluster 0:", estimator.estimate(X, cluster, expon))

### 4.3 Matchers

The initializer estimates every family on every cluster and fills a table of log-likelihoods. The matcher chooses the
assignment of clusters to families.

- `HungarianMatcher()` (the default) maximizes the total log-likelihood over all assignments. A pair that cannot be
  estimated — a cluster outside the support of a family, too few points, a degenerate component — is never chosen.
- `IdentityMatcher()` gives cluster `j` to family `j`. It is useful when the partitioner already knows the families,
  for example a quantile split of families listed from left to right.

In [ ]:
hungarian = ClusterInitializer(KMeansPartitioner(), matcher=HungarianMatcher()).initialize(X, families, random_state=42)
identity = ClusterInitializer(QuantilePartitioner(), matcher=IdentityMatcher()).initialize(X, families, random_state=42)
print("k-means + Hungarian:", round(hungarian.loglikelihood(X), 2))
show(hungarian)
print("quantile + identity:", round(identity.loglikelihood(X), 2))
show(identity)

### 4.4 The minimum size of a cluster

Both criteria of choice — the matcher and the multistart — prefer narrow clusters, because a narrow component has a
high likelihood. A cluster of a few outliers makes such a component, so by default every cluster must have at least
`2p` points, where `p` is the largest number of free parameters among the families (2 for `Normal`, so 4 points). An integer sets
the threshold explicitly, and `min_cluster_size=1` turns the check off.

In [ ]:
rng = np.random.default_rng(0)
with_outlier = np.concatenate([rng.normal(0, 1, 20), [100.0]])
try:
    ClusterInitializer(KMeansPartitioner()).initialize(with_outlier, [normal, normal], random_state=0)
except InitializationError as error:
    print("default threshold:", error)

lax = ClusterInitializer(KMeansPartitioner(), min_cluster_size=1).initialize(with_outlier, [normal, expon], random_state=0)
print("min_cluster_size=1:")
show(lax)

**Summary of Section 4:** the partitioner decides the clusters, the component estimator turns a cluster into
parameters, and the matcher decides which cluster goes to which family. The defaults — MLE and the Hungarian matcher —
fit most cases; the partitioner is always chosen explicitly.

## Section 5. Errors

Initializers raise two kinds of errors, and the difference matters.

- `InitializationError` (and its subclass `ComponentEstimationError`) is an unlucky draw: an empty or too small
  cluster, no assignment that gives every family a suitable cluster, a start that does not cover the sample. Another
  partition may succeed, so a multistart skips such runs. It is deliberately not a `ValueError`.
- Everything else is an error of the input, of the configuration or of the code — an empty `families`, too few points,
  an unsupported family — and another run would not fix it.

Random centers on the running example sometimes give a partition in which the exponential family fits no cluster.

In [ ]:
failures = []
for seed in range(20):
    try:
        random_initializer.initialize(X, families, random_state=seed)
    except InitializationError as error:
        failures.append(seed)
        last_error = error
print("failed seeds:", failures)
print("reason:", str(last_error)[:300])

try:
    random_initializer.initialize(X, [], random_state=0)
except ValueError as error:
    print("input error:", error)

assert failures and issubclass(ComponentEstimationError, InitializationError)

**Summary of Section 5:** catch `InitializationError` to retry with another random state; let every other
exception through. `MultiStartInitializer` does exactly that.

## Section 6. MultiStartInitializer

`MultiStartInitializer(initializer, n_init)` runs the nested initializer `n_init` times with independent children of
the generator (`rng.spawn(n_init)`) and returns the start with the largest `loglikelihood(X)`. Failed runs are
skipped; if some of them failed, one `RuntimeWarning` says how many and why; if all of them failed, it raises
`InitializationError` with the reason of the last one in `__cause__`.

In [ ]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    best = MultiStartInitializer(random_initializer, n_init=10).initialize(X, families, random_state=42)
for warning in caught:
    print("warning:", str(warning.message)[:200])
show(best)

single_scores = []
for child in np.random.default_rng(42).spawn(10):
    try:
        single_scores.append(random_initializer.initialize(X, families, random_state=child).loglikelihood(X))
    except InitializationError:
        pass
print("best single run:", round(max(single_scores), 2), " multistart:", round(best.loglikelihood(X), 2))

assert best.loglikelihood(X) == max(single_scores)

A multistart over a deterministic initializer (`QuantilePartitioner`, agglomerative clustering) is useless: every
run gives the same start.

**Summary of Section 6:** a multistart turns a random initializer into a more reliable one, and the integer seed of
the multistart still makes the whole result reproducible.

## Section 7. RefiningInitializer

The best start is not always the start that leads to the best fit. `RefiningInitializer(initializer, estimator,
*, fit_kwargs=None)` returns `estimator.fit(X, start, **fit_kwargs)`, and a multistart over it compares runs by the
result of the estimator. A short estimator gives the classic *emEM*: many short runs, then one long run from the
best of them. On this sample every reasonable start leads to the same fit; the example notebook
`notebooks/examples/initializers/local-optima.ipynb` shows a sample where emEM wins.

In [ ]:
short_em = em(20)
refined = RefiningInitializer(random_initializer, short_em)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", RuntimeWarning)
    start = MultiStartInitializer(refined, n_init=10).initialize(X, families, random_state=42)
result = em(100).fit(X, start)
show(result)
fig, ax = plt.subplots(figsize=(8, 3.5))
plot_mixtures(ax, X, {"emEM": result})
plt.show()

`fit_kwargs` go to every call of `fit`, for example the parameter groups of `ECM`. The names are checked against the
signature of `fit` when the initializer is created, so a typo fails at once and not after the first partition.

In [ ]:
ecm = ECM(breakpointers=[StepBreakpointer(20)], pruners=[], optimizer=ScipyNelderMead())
grouped = RefiningInitializer(
    ClusterInitializer(KMeansPartitioner()), ecm, fit_kwargs={"parameter_groups": {0: [{"mu"}, {"sigma"}]}}
)
show(grouped.initialize(X, families, random_state=42))

try:
    RefiningInitializer(ClusterInitializer(KMeansPartitioner()), em(20), fit_kwargs={"parameter_groups": {}})
except TypeError as error:
    print("checked at creation:", error)

An estimator with pruners may remove a component. The result would then have fewer components than families, so the
run is a failed one: `InitializationError`, which a multistart skips.

In [ ]:
rng = np.random.default_rng(42)
far_mode = np.concatenate([rng.normal(0, 1, 300), rng.normal(100, 1, 30)])
pruning = EM(breakpointers=[StepBreakpointer(2)], pruners=[PriorThresholdPruner(0.2)], optimizer=ScipyNelderMead())
try:
    RefiningInitializer(ClusterInitializer(KMeansPartitioner()), pruning).initialize(far_mode, [normal, normal], random_state=0)
except InitializationError as error:
    print(error)

**Summary of Section 7:** `RefiningInitializer` makes a start plus an estimator look like one initializer; wrapped in
a multistart, it selects runs by the fitted result.

## Section 8. Extending the API

### 8.1 A partitioner

A partitioner implements `partition(X, k, rng)` and returns a matrix `(n, k)` with one 1 in a row; all randomness comes
from `rng`. The example below splits the sample at known thresholds — domain knowledge instead of clustering. Its
columns go from left to right, so with families listed from left to right it pairs with `IdentityMatcher`. See the
example notebook `notebooks/examples/initializers/custom-partitioner.ipynb` for a full story.

In [ ]:
class ThresholdPartitioner(Partitioner):
    def __init__(self, thresholds: list[float]) -> None:
        self.thresholds = np.asarray(thresholds, dtype=np.float64)

    def partition(self, X: np.ndarray, k: int, rng: np.random.Generator) -> np.ndarray:
        if k != len(self.thresholds) + 1:
            raise ValueError(f"{len(self.thresholds)} thresholds make {len(self.thresholds) + 1} clusters, not {k}.")
        return np.eye(k)[np.searchsorted(self.thresholds, X)]


by_thresholds = ClusterInitializer(ThresholdPartitioner([-5.5, -1.0]), matcher=IdentityMatcher())
show(by_thresholds.initialize(X, families, random_state=0))

### 8.2 An initializer

An initializer of another kind implements `_initialize(X, families, rng)`. The common `initialize` does the rest: it
checks the input, creates the generator and checks the result — `k` components in the order of `families` and a
finite log-likelihood on the sample. A multistart works with it unchanged. The example below returns a mixture given by
hand, so the common checks catch a start that does not fit the families.

In [ ]:
class ManualInitializer(Initializer):
    def __init__(self, mixture: MixtureModel) -> None:
        self.mixture = mixture

    def _initialize(self, X, families, rng):
        return self.mixture


manual = MixtureModel([normal(mu=-8, sigma=1), normal(mu=-3, sigma=1), expon(lambda_=0.5)])
print(round(ManualInitializer(manual).initialize(X, families).loglikelihood(X), 2))
try:
    ManualInitializer(manual).initialize(X, [normal, expon, normal])
except ValueError as error:
    print("checked by the common initialize:", error)

**Summary of Section 8:** new partitioners, matchers and component estimators plug into `ClusterInitializer`; new
initializers inherit `Initializer` and get the checks and the multistart for free.

## Section 9. Final recap

- `initialize(X, families, random_state)` returns a start with one component per family, in the order of `families`;
  an integer seed makes it reproducible.
- `ClusterInitializer` = partitioner + component estimator + matcher; choose the partitioner, keep the defaults
  unless you know why.
- `InitializationError` is a bad draw: retry. Anything else is a mistake in the input: fix it.
- `MultiStartInitializer` keeps the best of several runs; around `RefiningInitializer` it compares runs by the result
  of an estimator.

The example notebooks in `notebooks/examples/initializers/` show these pieces on concrete problems.